# Agentic BM25 + embeddings on WANDS

This notebook mirrors `configs/agentic_ecom_2tools.yml` on the WANDS dataset. We implement a `SearchStrategy` that uses two tools (BM25 and embeddings) and an LLM to rank results.

Config values:
- model = gpt-5-mini
- search_tools = [bm25, embeddings]
- system_prompt = the agent prompt from the config file

Note: embedding search on the full WANDS corpus is expensive. You can cap `EMBEDDING_MAX_DOCS` to trade accuracy for speed.

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@73ae7d4cd80f
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)    # colab, share data across notebook runs on gdrive
# mount(use_gdrive=False) # <- colab without gdrive
# mount(use_gdrive=False, manual_path="/path/to/directory")  # <- force data path to specific directory, ie you're running locally.


## Get an OpenAI Key + load corpus

This will prompt you for an OpenAI Key to interact with GPT-5.

In [ ]:
import numpy as np
import pandas as pd

from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].to_numpy()
doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_id_lookup)}

(corpus.shape, judgments.shape)

## Build the BM25 tool

We build BM25 indices over title and description, then use the config boosts to score documents.

In [ ]:
from searcharray import SearchArray
from searcharray.similarity import bm25_similarity
from cheat_at_search.tokenizers import snowball_tokenizer

BM25_K1 = 1.2
BM25_B = 0.75
TITLE_BOOST = 9.3
DESCRIPTION_BOOST = 4.1

title_index = SearchArray.index(
    corpus["title"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)
description_index = SearchArray.index(
    corpus["description"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)

def bm25_tool(query, top_k=20):
    tokens = snowball_tokenizer(query)
    if not tokens:
        return [], []
    similarity = bm25_similarity(k1=BM25_K1, b=BM25_B)
    title_scores = title_index.score(tokens, similarity=similarity)
    description_scores = description_index.score(tokens, similarity=similarity)
    scores = TITLE_BOOST * title_scores + DESCRIPTION_BOOST * description_scores
    top_k = min(top_k, len(scores))
    if top_k == 0:
        return [], []
    top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
    top_sorted = top_idx[np.argsort(-scores[top_idx])]
    doc_ids = doc_id_lookup[top_sorted].tolist()
    return doc_ids, scores[top_sorted].tolist()

## Build the embeddings tool

We compute embeddings for the corpus once, then use cosine similarity for retrieval. This is the most expensive step; reduce `EMBEDDING_MAX_DOCS` if needed.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_BATCH = 128
EMBEDDING_MAX_DOCS = None  # set to e.g. 5000 for faster iteration

def _row_text(row):
    title = row.get("title")
    description = row.get("description")
    title_text = title.strip() if isinstance(title, str) else ""
    description_text = description.strip() if isinstance(description, str) else ""
    if title_text and description_text:
        return f"{title_text}\n\n{description_text}"
    if title_text:
        return title_text
    return description_text

def embed_texts(texts):
    embeddings = []
    for start in range(0, len(texts), EMBEDDING_BATCH):
        batch = texts[start : start + EMBEDDING_BATCH]
        resp = openai.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        embeddings.extend([item.embedding for item in resp.data])
    return np.asarray(embeddings, dtype=np.float32)

emb_corpus = corpus if EMBEDDING_MAX_DOCS is None else corpus.head(EMBEDDING_MAX_DOCS)
emb_doc_ids = emb_corpus["doc_id"].to_numpy()
emb_texts = [_row_text(row) for _, row in emb_corpus.iterrows()]

embeddings = embed_texts(emb_texts)
embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)

def embeddings_tool(query, top_k=20):
    query_emb = embed_texts([query])[0]
    query_emb = query_emb / (np.linalg.norm(query_emb) + 1e-12)
    scores = embeddings @ query_emb
    top_k = min(top_k, len(scores))
    if top_k == 0:
        return [], []
    top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
    top_sorted = top_idx[np.argsort(-scores[top_idx])]
    doc_ids = emb_doc_ids[top_sorted].tolist()
    return doc_ids, scores[top_sorted].tolist()

## Implement the agentic SearchStrategy

The agent receives BM25 + embedding candidates and returns an ordered list of doc_ids. We then map doc_ids back to corpus indices.

In [ ]:
import json
from cheat_at_search.strategy.strategy import SearchStrategy

SYSTEM_PROMPT = (
    "You take user search queries and use a search tool to find products.\n\n"
    "Look at the search tools you have, their limitations, how they work, etc when forming your plan.\n\n"
    "Finally return results to the user per the SearchResults schema, ranked best to worst.\n\n"
    "Gather results until you have 10 best matches you can find. It's important to return at least 10.\n\n"
    "It's very important you consider carefully the correct ranking as you'll be evaluated on\n"
    "how close that is to the average shoppers ideal ranking."
)

def _candidate_text(doc_id):
    row = corpus.iloc[doc_id_to_index[doc_id]]
    title = row.get("title") or ""
    description = row.get("description") or ""
    title = title.strip()
    description = description.strip()
    snippet = description[:240].replace("\n", " ")
    return f"{doc_id} | {title} | {snippet}"

def rank_with_llm(query, candidates):
    candidate_lines = "\n".join([_candidate_text(doc_id) for doc_id in candidates])
    user_prompt = (
        "You are ranking e-commerce search results.\n"
        f"Query: {query}\n\n"
        "Candidates (doc_id | title | snippet):\n"
        f"{candidate_lines}\n\n"
        "Return JSON with the schema: {\"ranked_results\": [doc_id, ...]}"
    )
    response = openai.responses.create(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )
    text = response.output_text
    try:
        data = json.loads(text)
        return data.get("ranked_results", [])
    except json.JSONDecodeError:
        return []

class AgenticSearchStrategy(SearchStrategy):
    def __init__(self, corpus, top_k=10, workers=1):
        super().__init__(corpus, top_k=top_k, workers=workers)

    def search(self, query, k=10):
        bm25_doc_ids, _ = bm25_tool(query, top_k=20)
        emb_doc_ids, _ = embeddings_tool(query, top_k=20)
        candidates = list(dict.fromkeys(bm25_doc_ids + emb_doc_ids))
        ranked_doc_ids = rank_with_llm(query, candidates)

        # Ensure we return at least k results, filling with BM25 if needed.
        if len(ranked_doc_ids) < k:
            for doc_id in bm25_doc_ids:
                if doc_id not in ranked_doc_ids:
                    ranked_doc_ids.append(doc_id)
                if len(ranked_doc_ids) >= k:
                    break

        ranked_doc_ids = ranked_doc_ids[:k]
        indices = [doc_id_to_index[doc_id] for doc_id in ranked_doc_ids if doc_id in doc_id_to_index]
        return indices, [1.0] * len(indices)

## Run the strategy and evaluate

We run the agentic strategy over the judgment queries and summarize NDCG/MRR. Caching is disabled for notebook clarity.

In [ ]:
from cheat_at_search.search import run_strategy

NUM_QUERIES = None  # set to e.g. 50 for faster iteration

agentic = AgenticSearchStrategy(corpus, top_k=10, workers=1)
graded = run_strategy(
    agentic,
    judgments,
    num_queries=NUM_QUERIES,
    seed=42,
    show_progress=True,
    cache=False,
)

graded.head()

## Summarize metrics

In [ ]:
from cheat_at_search.search import ndcgs, mrrs

ndcg_series = ndcgs(graded)
mrr_series = mrrs(graded)

pd.DataFrame(
    {
        "metric": ["NDCG@10", "MRR"],
        "value": [ndcg_series.mean(), mrr_series.mean()],
    }
)